In [19]:
library(tidyverse)

In [20]:
# 1.Read the dataset
housing <- read_csv("housing.csv")

Rows: 20640 Columns: 10
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): ocean_proximity
dbl (9): longitude, latitude, housing_median_age, total_rooms, total_bedroom...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [21]:
# 2.Basic checks
dim(housing)
names(housing)
glimpse(housing)

[1] 20640    10

[1] "longitude"          "latitude"           "housing_median_age"
 [4] "total_rooms"        "total_bedrooms"     "population"        
 [7] "households"         "median_income"      "median_house_value"
[10] "ocean_proximity"

Rows: 20,640
Columns: 10
$ longitude          <dbl> -122.23, -122.22, -122.24, -122.25, -122.25, -122.2…
$ latitude           <dbl> 37.88, 37.86, 37.85, 37.85, 37.85, 37.85, 37.84, 37…
$ housing_median_age <dbl> 41, 21, 52, 52, 52, 52, 52, 52, 42, 52, 52, 52, 52,…
$ total_rooms        <dbl> 880, 7099, 1467, 1274, 1627, 919, 2535, 3104, 2555,…
$ total_bedrooms     <dbl> 129, 1106, 190, 235, 280, 213, 489, 687, 665, 707, …
$ population         <dbl> 322, 2401, 496, 558, 565, 413, 1094, 1157, 1206, 15…
$ households         <dbl> 126, 1138, 177, 219, 259, 193, 514, 647, 595, 714, …
$ median_income      <dbl> 8.3252, 8.3014, 7.2574, 5.6431, 3.8462, 4.0368, 3.6…
$ median_house_value <dbl> 452600, 358500, 352100, 341300, 342200, 269700, 299…
$ ocean_proximity    <chr> "NEAR BAY", "NEAR BAY", "NEAR BAY", "NEAR BAY", "NE…


In [22]:
# 3.Changes data type
housing <- housing |>
  mutate(
    ocean_proximity = factor(ocean_proximity)
  )

In [24]:
# 4.Number of missing values
missing_summary <- housing |>
  summarise(across(everything(), ~ sum(is.na(.)))) |>
  pivot_longer(
    cols = everything(),
    names_to = "variable",
    values_to = "n_missing"
  ) |>
  mutate(
    pct_missing = round(100 * n_missing / nrow(housing), 2)
  ) |>
  arrange(desc(n_missing))

missing_summary

variable,n_missing,pct_missing
<chr>,<int>,<dbl>
total_bedrooms,207,1
longitude,0,0
latitude,0,0
housing_median_age,0,0
total_rooms,0,0
population,0,0
households,0,0
median_income,0,0
median_house_value,0,0


In [26]:
# 5.Check Missing values for every rows : !!!Important: whether missingness is small or concerning
housing |> 
  mutate(row_has_missing = if_any(everything(), is.na)) |> 
  summarise(
    n_rows = n(),
    rows_with_missing = sum(row_has_missing),
    pct_rows_with_missing = round(100 * mean(row_has_missing), 2)
  )

n_rows,rows_with_missing,pct_rows_with_missing
<int>,<int>,<dbl>
20640,207,1


In [27]:
# 6.Distribution of ocean proximity
housing |>
  count(ocean_proximity) |>
  mutate(
    prop = round(n / sum(n), 4)
  )

ocean_proximity,n,prop
<fct>,<int>,<dbl>
<1H OCEAN,9136,0.4426
INLAND,6551,0.3174
ISLAND,5,0.0002
NEAR BAY,2290,0.1109
NEAR OCEAN,2658,0.1288


In [28]:
# 7.Numerical summaries
housing |>
  select(where(is.numeric)) |>
  summary()

   longitude         latitude     housing_median_age  total_rooms   
 Min.   :-124.3   Min.   :32.54   Min.   : 1.00      Min.   :    2  
 1st Qu.:-121.8   1st Qu.:33.93   1st Qu.:18.00      1st Qu.: 1448  
 Median :-118.5   Median :34.26   Median :29.00      Median : 2127  
 Mean   :-119.6   Mean   :35.63   Mean   :28.64      Mean   : 2636  
 3rd Qu.:-118.0   3rd Qu.:37.71   3rd Qu.:37.00      3rd Qu.: 3148  
 Max.   :-114.3   Max.   :41.95   Max.   :52.00      Max.   :39320  
                                                                    
 total_bedrooms     population      households     median_income    
 Min.   :   1.0   Min.   :    3   Min.   :   1.0   Min.   : 0.4999  
 1st Qu.: 296.0   1st Qu.:  787   1st Qu.: 280.0   1st Qu.: 2.5634  
 Median : 435.0   Median : 1166   Median : 409.0   Median : 3.5348  
 Mean   : 537.9   Mean   : 1425   Mean   : 499.5   Mean   : 3.8707  
 3rd Qu.: 647.0   3rd Qu.: 1725   3rd Qu.: 605.0   3rd Qu.: 4.7432  
 Max.   :6445.0   Max.   :35682   

In [29]:
# 8.Response variable summary
housing |>
  summarise(
    min_value = min(median_house_value, na.rm = TRUE),
    q1 = quantile(median_house_value, 0.25, na.rm = TRUE),
    median = median(median_house_value, na.rm = TRUE),
    mean = mean(median_house_value, na.rm = TRUE),
    q3 = quantile(median_house_value, 0.75, na.rm = TRUE),
    max_value = max(median_house_value, na.rm = TRUE),
    sd_value = sd(median_house_value, na.rm = TRUE)
  )

min_value,q1,median,mean,q3,max_value,sd_value
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
14999,119600,179700,206855.8,264725,500001,115395.6


In [30]:
# 9.Check the cap at 500001
housing |>
  summarise(
    capped_n = sum(median_house_value == 500001, na.rm = TRUE),
    capped_pct = round(100 * mean(median_house_value == 500001, na.rm = TRUE), 2)
  )

capped_n,capped_pct
<int>,<dbl>
965,4.68


In [32]:
# 10.Key grouped summary for your research question
housing |>
  group_by(ocean_proximity) |>
  summarise(
    n = n(),
    mean_house_value = mean(median_house_value, na.rm = TRUE),
    median_house_value_group = median(median_house_value, na.rm = TRUE),
    sd_house_value = sd(median_house_value, na.rm = TRUE),
    mean_income = mean(median_income, na.rm = TRUE),
    .groups = "drop"
  ) |>
  arrange(desc(median_house_value_group))

ocean_proximity,n,mean_house_value,median_house_value_group,sd_house_value,mean_income
<fct>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
ISLAND,5,380440.0,414700,80559.56,2.744420
NEAR BAY,2290,259212.3,233800,122818.54,4.172885
NEAR OCEAN,2658,249434.0,229450,122477.15,4.005785
<1H OCEAN,9136,240084.3,214850,106124.29,4.230682
INLAND,6551,124805.4,108500,70007.91,3.208996
